In [1]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [2]:
data = pd.DataFrame({
    "CGPA": [6.5, 6.7, 7.0, 7.5, 8.0],
    "Placement": [0, 0, 0, 1, 1]
})

In [4]:
# Number of positive and negative observations
positive = (data["Placement"] == 1).sum()
negative = (data["Placement"] == 0).sum()

p_initial = positive / len(data)

print("\nInitial Probability P(Y=1):", p_initial)


Initial Probability P(Y=1): 0.4


In [5]:
F0 = np.log(p_initial / (1 - p_initial))

print("Initial Raw Score F0:", F0)

Initial Raw Score F0: -0.4054651081081643


In [6]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


initial_probability = sigmoid(F0)

print("Probability after Sigmoid:", initial_probability)

Probability after Sigmoid: 0.4


In [7]:
data["Raw_Score"] = F0

data["Probability"] = sigmoid(data["Raw_Score"])

# Gradient: G = p - y
data["Gradient_G"] = (
    data["Probability"] - data["Placement"]
)

# Hessian: H = p(1-p)
data["Hessian_H"] = (
    data["Probability"]
    * (1 - data["Probability"])
)

print("\nInitial Gradient and Hessian:")
print(data)



Initial Gradient and Hessian:
   CGPA  Placement  Raw_Score  Probability  Gradient_G  Hessian_H
0   6.5          0  -0.405465          0.4         0.4       0.24
1   6.7          0  -0.405465          0.4         0.4       0.24
2   7.0          0  -0.405465          0.4         0.4       0.24
3   7.5          1  -0.405465          0.4        -0.6       0.24
4   8.0          1  -0.405465          0.4        -0.6       0.24


In [8]:
lambda_reg = 0

G_root = data["Gradient_G"].sum()
H_root = data["Hessian_H"].sum()

root_score = (
    G_root ** 2
    / (H_root + lambda_reg)
)

print("\nRoot Node:")
print("G =", G_root)
print("H =", H_root)
print("Root Score =", root_score)


Root Node:
G = 2.220446049250313e-16
H = 1.2
Root Score = 4.1086505480261033e-32


In [9]:
data = data.sort_values("CGPA").reset_index(drop=True)

cgpa = data["CGPA"].values

candidate_splits = [
    (cgpa[i] + cgpa[i + 1]) / 2
    for i in range(len(cgpa) - 1)
]

print("\nCandidate Splits:")
print(candidate_splits)



Candidate Splits:
[np.float64(6.6), np.float64(6.85), np.float64(7.25), np.float64(7.75)]


In [10]:
def calculate_score(G, H, lambda_reg=0):
    return (G ** 2) / (H + lambda_reg)


split_results = []

for split in candidate_splits:

    left = data[data["CGPA"] < split]
    right = data[data["CGPA"] >= split]

    G_left = left["Gradient_G"].sum()
    H_left = left["Hessian_H"].sum()

    G_right = right["Gradient_G"].sum()
    H_right = right["Hessian_H"].sum()

    score_left = calculate_score(
        G_left,
        H_left,
        lambda_reg
    )

    score_right = calculate_score(
        G_right,
        H_right,
        lambda_reg
    )

    gain = 0.5 * (
        score_left
        + score_right
        - root_score
    )

    split_results.append({
        "Split": split,
        "G_Left": G_left,
        "H_Left": H_left,
        "G_Right": G_right,
        "H_Right": H_right,
        "Gain": gain
    })


split_results = pd.DataFrame(split_results)

print("\nSplit Evaluation:")
print(split_results)


Split Evaluation:
   Split  G_Left  H_Left  G_Right  H_Right      Gain
0   6.60     0.4    0.24     -0.4     0.96  0.416667
1   6.85     0.8    0.48     -0.8     0.72  1.111111
2   7.25     1.2    0.72     -1.2     0.48  2.500000
3   7.75     0.6    0.96     -0.6     0.24  0.937500


In [11]:
best_split = split_results.loc[
    split_results["Gain"].idxmax()
]

print("\nBest Split:")
print(best_split)




Best Split:
Split      7.25
G_Left     1.20
H_Left     0.72
G_Right   -1.20
H_Right    0.48
Gain       2.50
Name: 2, dtype: float64


In [12]:
best_value = best_split["Split"]

left = data[data["CGPA"] < best_value]
right = data[data["CGPA"] >= best_value]

G_left = left["Gradient_G"].sum()
H_left = left["Hessian_H"].sum()

G_right = right["Gradient_G"].sum()
H_right = right["Hessian_H"].sum()

left_output = -G_left / (
    H_left + lambda_reg
)

right_output = -G_right / (
    H_right + lambda_reg
)

print("\nLeaf Outputs:")

print("Left Leaf :", left_output)
print("Right Leaf:", right_output)


Leaf Outputs:
Left Leaf : -1.666666666666667
Right Leaf: 2.5


In [13]:
learning_rate = 0.3

data["Tree_Output"] = np.where(
    data["CGPA"] < best_value,
    left_output,
    right_output
)

data["Updated_Raw_Score"] = (
    data["Raw_Score"]
    + learning_rate * data["Tree_Output"]
)

In [14]:
data["Updated_Probability"] = sigmoid(
    data["Updated_Raw_Score"]
)

print("\nAfter Tree 1:")
print(
    data[
        [
            "CGPA",
            "Placement",
            "Tree_Output",
            "Updated_Raw_Score",
            "Updated_Probability"
        ]
    ]
)



After Tree 1:
   CGPA  Placement  Tree_Output  Updated_Raw_Score  Updated_Probability
0   6.5          0    -1.666667          -0.905465             0.287929
1   6.7          0    -1.666667          -0.905465             0.287929
2   7.0          0    -1.666667          -0.905465             0.287929
3   7.5          1     2.500000           0.344535             0.585292
4   8.0          1     2.500000           0.344535             0.585292


In [15]:
data["Prediction"] = np.where(
    data["Updated_Probability"] >= 0.5,
    1,
    0
)

print("\nPredictions after Tree 1:")
print(
    data[
        [
            "CGPA",
            "Placement",
            "Updated_Probability",
            "Prediction"
        ]
    ]
)


Predictions after Tree 1:
   CGPA  Placement  Updated_Probability  Prediction
0   6.5          0             0.287929           0
1   6.7          0             0.287929           0
2   7.0          0             0.287929           0
3   7.5          1             0.585292           1
4   8.0          1             0.585292           1


In [16]:
X = data[["CGPA"]]
y = data["Placement"]

model = XGBClassifier(
    n_estimators=3,
    learning_rate=0.3,
    max_depth=2,
    reg_lambda=0,
    reg_alpha=0,
    tree_method="hist",
    device="cpu",
    objective="binary:logistic",
    random_state=42
)

model.fit(X, y)



,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cpu'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XG

In [17]:
xgb_probability = model.predict_proba(X)[:, 1]
xgb_prediction = model.predict(X)

comparison = pd.DataFrame({
    "CGPA": X["CGPA"],
    "Actual": y,
    "XGB_Probability": xgb_probability,
    "XGB_Prediction": xgb_prediction
})

print("\nActual XGBoost Predictions:")
print(comparison)


Actual XGBoost Predictions:
   CGPA  Actual  XGB_Probability  XGB_Prediction
0   6.5       0              0.4               0
1   6.7       0              0.4               0
2   7.0       0              0.4               0
3   7.5       1              0.4               0
4   8.0       1              0.4               0


In [18]:
accuracy = accuracy_score(
    y,
    xgb_prediction
)

print("\nXGBoost Accuracy:", accuracy)




XGBoost Accuracy: 0.6


In [19]:
trees = model.get_booster().get_dump()

for i, tree in enumerate(trees):

    print(
        f"\n TREE {i + 1}"
    )

    print(tree)


 TREE 1
0:leaf=7.4505806e-09


 TREE 2
0:leaf=7.4505806e-09


 TREE 3
0:leaf=7.4505806e-09

